
# **FuzzyDiff: Fuzzy Logical Diffusion for Structuredand Graded Image Generation**

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Optional, Union
import torch
from PIL import Image
from torch.nn.functional import interpolate
import torch.nn.functional as F
import numpy as np
import gc
import abc

In [ ]:
!pip install git+https://github.com/openai/CLIP.git
import clip

## Configuration

In [ ]:
@dataclass
class RunConfig:
    prompt: str = "a black dog and a white cat"
    seeds: List[int] = field(default_factory=lambda: [84])
    output_path: Path = Path('./outputs')
    n_inference_steps: int = 50
    guidance_scale: float = 9.5
    height: int = 768           
    width: int = 768           
    attn_res: int = 16
    max_iter_to_alter: int = 25
    attend_excite_lr: float = 0.2
    alpha: float = 10.0
    spatial_loss_weight: float = 0.5
    token_indices: List[int] = field(default_factory=list)

    def __post_init__(self):
        self.output_path.mkdir(parents=True, exist_ok=True)

print("✓ Config created")

In [ ]:
def soft_truth(attn_map: torch.Tensor, alpha: float = 10.0) -> torch.Tensor:

    attn_flat = attn_map.flatten()
    weights   = torch.softmax(attn_flat * alpha, dim=0)
    return (weights * attn_flat).sum()


def predicate_truth(
    attention_map: torch.Tensor,
    token_idx: int,
    alpha: float = 10.0
) -> torch.Tensor:
    
    if token_idx >= attention_map.shape[-1]:
        return torch.tensor(0.0, device=attention_map.device)
    token_attn = attention_map[..., token_idx]
    return soft_truth(token_attn, alpha)


def soft_left(attn_a: torch.Tensor, attn_b: torch.Tensor) -> torch.Tensor:

    h, w     = attn_a.shape
    x        = torch.linspace(0, 1, w, device=attn_a.device)
    center_a = (attn_a * x).sum() / (attn_a.sum() + 1e-8)
    center_b = (attn_b * x).sum() / (attn_b.sum() + 1e-8)
    return torch.sigmoid(10 * (center_b - center_a))


def soft_under(attn_a: torch.Tensor, attn_b: torch.Tensor) -> torch.Tensor:

    h, w     = attn_a.shape
    y        = torch.linspace(0, 1, h, device=attn_a.device).view(h, 1)
    center_a = (attn_a * y).sum() / (attn_a.sum() + 1e-8)
    center_b = (attn_b * y).sum() / (attn_b.sum() + 1e-8)
    return torch.sigmoid(10 * (center_a - center_b))


    Combines:
        1. Presence loss  — penalises tokens with low attention
        2. Spatial loss   — penalises spatial predicate violations

In [ ]:
def compute_fuzzy_loss(
    avg_attn: Dict[str, torch.Tensor],
    token_indices: List[int],
    alpha: float = 10.0,
    spatial_loss_weight: float = 0.5,
) -> torch.Tensor:

    cross_maps = []
    for key in ["down_cross", "mid_cross", "up_cross"]:
        if key in avg_attn and avg_attn[key].numel() > 0:
            cross_maps.append(avg_attn[key])   # (B, N, T)

    if len(cross_maps) == 0:
        return torch.tensor(0.0, requires_grad=True)

    attn_map = cross_maps[0]   # (B, N, T)

    # ── 1. Presence loss ─────────────────────────────────────────
    token_scores = []
    for idx in token_indices:
        score = predicate_truth(attn_map, idx, alpha)
        token_scores.append(score)
        print(f"      predicate_truth(token={idx}): {score.item():.4f}")

    if len(token_scores) == 0:
        return torch.tensor(0.0, requires_grad=True)

    stacked       = torch.stack(token_scores)
    min_score     = stacked.min()
    presence_loss = -torch.log(min_score + 1e-8)

    # ── 2. Spatial loss ──────────────────────────────────────────
    spatial_loss = torch.tensor(0.0, device=attn_map.device)

    if len(token_indices) >= 2:
        N       = attn_map.shape[1]
        spatial = int(N ** 0.5)

        if spatial * spatial == N:
            for i in range(len(token_indices) - 1):
                idx_a = token_indices[i]
                idx_b = token_indices[i + 1]

                if idx_a >= attn_map.shape[-1] or idx_b >= attn_map.shape[-1]:
                    continue

                map_a = attn_map[0, :, idx_a].view(spatial, spatial)
                map_b = attn_map[0, :, idx_b].view(spatial, spatial)

                left_score  = soft_left(map_a, map_b)
                under_score = soft_under(map_a, map_b)

                spatial_loss = spatial_loss - spatial_loss_weight * (
                    torch.log(left_score + 1e-8) +
                    torch.log(under_score + 1e-8)
                )

    total_loss = presence_loss + spatial_loss
    return total_loss


print("✓ Fuzzy predicate utilities created")


In [ ]:
clip_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=clip_device)
clip_model.eval()
print("✓ CLIP loaded")

In [ ]:
class AttentionControl(abc.ABC):
    def __init__(self):
        self.cur_step              = 0
        self.num_att_layers        = -1
        self.cur_att_layer         = 0
        self.num_uncond_att_layers = 0

    @abc.abstractmethod
    def forward(self, attn, is_cross: bool, place_in_unet: str):
        raise NotImplementedError

    def __call__(self, attn, is_cross: bool, place_in_unet: str):
        attn_processed = self.forward(attn, is_cross, place_in_unet)
        self.cur_att_layer += 1
        if self.cur_att_layer == self.num_att_layers + self.num_uncond_att_layers:
            self.cur_att_layer = 0
            self.cur_step     += 1
            self.between_steps()
        return attn_processed

    def between_steps(self):
        return

    def reset(self):
        self.cur_step      = 0
        self.cur_att_layer = 0

In [ ]:
class AttentionStore(AttentionControl):

    @staticmethod
    def get_empty_store():
        return {
            "down_cross": [], "mid_cross": [], "up_cross": [],
            "down_self":  [], "mid_self":  [], "up_self":  [],
        }

    def __init__(self, attn_res=16):
        super().__init__()
        self.attn_res        = attn_res
        self.step_store      = self.get_empty_store()
        self.attention_store = {}

    def forward(self, attn, is_cross: bool, place_in_unet: str):
        key     = f"{place_in_unet}_{'cross' if is_cross else 'self'}"
        # attn shape: (B * num_heads, N, T) — SDXL uses multi-head attention
        # Reshape to (B, num_heads, N, T) then average over heads → (1, N, T)
        BH, N, T = attn.shape
        size     = self.attn_res

        # Infer batch size: assume batch=1 (single image generation)
        # BH = B * num_heads; we don't know num_heads directly, but we store
        # head-averaged maps so shapes are always (1, N, T) regardless of heads.
        attn_avg = attn.mean(0, keepdim=True)   # (1, N, T) — average over all heads

        if N > size ** 2:
            h = w = int(N ** 0.5)
            attn_reshaped = attn_avg.view(1, h, w, T).permute(0, 3, 1, 2)   # (1, T, h, w)
            attn_down     = interpolate(
                attn_reshaped, size=(size, size),
                mode="bilinear", align_corners=False
            )
            attn_down = attn_down.permute(0, 2, 3, 1).reshape(1, size * size, T)  # (1, N', T)
            self.step_store[key].append(attn_down.detach())
        else:
            self.step_store[key].append(attn_avg.detach())

        return attn

    def between_steps(self):
        if len(self.attention_store) == 0:
            self.attention_store = self.get_empty_store()
        for key in self.step_store:
            self.attention_store[key].extend(self.step_store[key])
        self.step_store = self.get_empty_store()

    def get_average_attention(self):
        avg_attn = {}
        for key in self.attention_store:
            if len(self.attention_store[key]) > 0:
                stacked       = torch.stack(self.attention_store[key])
                avg_attn[key] = stacked.mean(0)
        return avg_attn

    def get_current_step_attention(self):
        """Read from step_store BEFORE between_steps() is called."""
        cur_attn = {}
        for key in self.step_store:
            if len(self.step_store[key]) > 0:
                stacked       = torch.stack(self.step_store[key])
                cur_attn[key] = stacked.mean(0)
        return cur_attn

    def reset(self):
        super().reset()
        self.step_store      = self.get_empty_store()
        self.attention_store = {}


print("✓ Attention control created")

SDXL UNet uses Transformer2DModel blocks whose attention layers expose a
slightly different interface.  We store (is_cross, place) so the processor
itself decides what to capture, identical logic to the SD-1.5 version but
tolerant of the extra kwargs SDXL passes (added_cond_kwargs, etc.).

In [ ]:
class AttendExciteAttnProcessor:
    def __init__(self, attnstore, place_in_unet):
        self.attnstore     = attnstore
        self.place_in_unet = place_in_unet

    def __call__(
        self, attn, hidden_states,
        encoder_hidden_states=None, attention_mask=None,
        temb=None, **kwargs          # ← absorb SDXL extra kwargs
    ):
        residual   = hidden_states
        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(
                batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape
            if encoder_hidden_states is None
            else encoder_hidden_states.shape
        )

        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(
                attention_mask, sequence_length, batch_size)

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(
                hidden_states.transpose(1, 2)).transpose(1, 2)

        query    = attn.to_q(hidden_states)
        is_cross = encoder_hidden_states is not None

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(
                encoder_hidden_states)

        key   = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        query = attn.head_to_batch_dim(query)
        key   = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)

        attention_probs = attn.get_attention_scores(query, key, attention_mask)

        # Store conditional cross-attention only (skip uncond half for CFG)
        if is_cross:
            if attention_probs.shape[0] == 2 * batch_size:
                attn_to_store = attention_probs[batch_size:]
            else:
                attn_to_store = attention_probs
            self.attnstore(attn_to_store, is_cross, self.place_in_unet)

        hidden_states = torch.bmm(attention_probs, value)
        hidden_states = attn.batch_to_head_dim(hidden_states)
        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(
                batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor
        return hidden_states


In [ ]:
from diffusers import DiffusionPipeline, StableDiffusionXLPipeline
from diffusers import AutoencoderKL
from diffusers import DPMSolverMultistepScheduler

In [ ]:
class FuzzyAttendExciteSDXLPipeline(StableDiffusionXLPipeline):

    def register_attention_control(self, controller):
        """Hook custom attention processor into every UNet attention layer."""
        attn_procs = {}
        for name in self.unet.attn_processors.keys():
            if "mid" in name:
                place = "mid"
            elif "up" in name:
                place = "up"
            else:
                place = "down"
            attn_procs[name] = AttendExciteAttnProcessor(controller, place)
        self.unet.set_attn_processor(attn_procs)
        controller.num_att_layers = len(attn_procs)

    def get_token_attention(self, attention_store, token_indices):
        """Extract peak attention score per token across all locations."""
        avg_attn = attention_store.get_average_attention()
        scores   = {}
        for idx in token_indices:
            token_scores = []
            for location in ["down_cross", "mid_cross", "up_cross"]:
                if location in avg_attn and avg_attn[location].numel() > 0:
                    attn_map = avg_attn[location]
                    if idx < attn_map.shape[-1]:
                        token_attn = attn_map[..., idx]
                        token_scores.append(token_attn.max().item())
            scores[idx] = max(token_scores) if token_scores else 0.0
        return scores

    def _update_latents_with_fuzzy_loss(
        self,
        latents: torch.Tensor,
        attention_store: AttentionStore,
        token_indices: List[int],
        cfg: RunConfig,
    ) -> torch.Tensor:
        """Attend-Excite latent update using fuzzy predicates."""
        latents  = latents.detach().requires_grad_(True)
        cur_attn = attention_store.get_current_step_attention()

        if len(cur_attn) == 0:
            return latents.detach()

        loss = compute_fuzzy_loss(
            avg_attn            = cur_attn,
            token_indices       = token_indices,
            alpha               = cfg.alpha,
            spatial_loss_weight = cfg.spatial_loss_weight,
        )

        if loss.requires_grad:
            grad = torch.autograd.grad(
                outputs      = loss,
                inputs       = latents,
                allow_unused = True
            )[0]

            if grad is not None:
                with torch.no_grad():
                    latents = latents - cfg.attend_excite_lr * grad
            else:
                print("    [WARN] Gradient is None — attention graph not connected to latents")

        return latents.detach()

    @torch.no_grad()
    def __call__(
        self,
        prompt: Union[str, List[str]],
        token_indices: List[int]             = None,
        attention_store: AttentionStore      = None,
        height: int                          = 768,
        width: int                           = 768,
        num_inference_steps: int             = 50,
        guidance_scale: float                = 9.5,
        generator: Optional[torch.Generator] = None,
        output_type: str                     = "pil",
        cfg: RunConfig                       = None,
        negative_prompt: Optional[str]       = None,
        **kwargs
    ):

        exec_device = self._execution_device

        if isinstance(prompt, str):
            prompt = [prompt]

        batch_size = len(prompt)
        (
            prompt_embeds,
            negative_prompt_embeds,
            pooled_prompt_embeds,
            negative_pooled_prompt_embeds,
        ) = self.encode_prompt(
            prompt            = prompt,
            device            = exec_device,
            num_images_per_prompt = 1,
            do_classifier_free_guidance = guidance_scale > 1.0,
            negative_prompt   = [negative_prompt or ""] * batch_size,
        )

        # ── Prepare SDXL add_time_ids conditioning ───────────────────
        original_size    = (height, width)
        target_size      = (height, width)
        crops_top_left   = (0, 0)

        # Derive text_encoder_projection_dim from UNet's add_embedding layer —
        # text_encoder_2.config.projection_dim is None for this checkpoint.
        try:
            unet_in  = self.unet.add_embedding.linear_1.in_features   # 2816
            time_dim = self.unet.config.addition_time_embed_dim        # 256
            text_encoder_projection_dim = unet_in - time_dim * 6      # 1280
        except Exception:
            text_encoder_projection_dim = 1280

        add_time_ids = self._get_add_time_ids(
            original_size,
            crops_top_left,
            target_size,
            dtype                       = prompt_embeds.dtype,
            text_encoder_projection_dim = text_encoder_projection_dim,
        ).to(exec_device)

        # Repeat for batch
        add_time_ids = add_time_ids.repeat(batch_size, 1)

        if guidance_scale > 1.0:
            negative_add_time_ids = add_time_ids
            add_time_ids          = torch.cat([negative_add_time_ids, add_time_ids], dim=0)
            prompt_embeds         = torch.cat([negative_prompt_embeds, prompt_embeds], dim=0)
            pooled_prompt_embeds  = torch.cat([negative_pooled_prompt_embeds, pooled_prompt_embeds], dim=0)

        # ── Prepare latents ──────────────────────────────────────────
        latents = torch.randn(
            (batch_size, self.unet.config.in_channels,
             height // 8, width // 8),
            generator = generator,
            device    = exec_device,
            dtype     = prompt_embeds.dtype,
        )
        self.scheduler.set_timesteps(num_inference_steps, device=exec_device)
        latents = latents * self.scheduler.init_noise_sigma

        max_iter = cfg.max_iter_to_alter if cfg else 25

        # ── Denoising loop ───────────────────────────────────────────
        for i, t in enumerate(self.scheduler.timesteps):

            latent_model_input = torch.cat([latents] * 2) if guidance_scale > 1.0 else latents
            latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)

            # SDXL UNet requires added_cond_kwargs
            added_cond_kwargs = {
                "text_embeds": pooled_prompt_embeds,
                "time_ids":    add_time_ids,
            }

            noise_pred = self.unet(
                latent_model_input,
                t,
                encoder_hidden_states = prompt_embeds,
                added_cond_kwargs     = added_cond_kwargs,
                return_dict           = False,
            )[0]

            # Classifier-free guidance
            if guidance_scale > 1.0:
                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                noise_pred = noise_pred_uncond + guidance_scale * (
                    noise_pred_text - noise_pred_uncond)

            latents = self.scheduler.step(noise_pred, t, latents).prev_sample

            # ── FUZZY ───────────────────────────────
            if attention_store is not None and token_indices and i < max_iter:
                print(f"\n  [Step {i+1}/{max_iter}] Applying fuzzy loss...")
                with torch.enable_grad():
                    latents = self._update_latents_with_fuzzy_loss(
                        latents         = latents,
                        attention_store = attention_store,
                        token_indices   = token_indices,
                        cfg             = cfg if cfg else RunConfig(),
                    )
                attention_store.between_steps()
            else:
                if attention_store is not None:
                    attention_store.between_steps()

        # ── Decode latents → image ───────────────────────────────────
        # SDXL VAE scaling factor is 0.13025 (not 0.18215)
        latents = latents / self.vae.config.scaling_factor
        image   = self.vae.decode(latents, return_dict=False)[0]
        image   = (image / 2 + 0.5).clamp(0, 1)
        image   = image.cpu().permute(0, 2, 3, 1).float().numpy()

        if output_type == "pil":
            image = self.numpy_to_pil(image)

        return image


# ===============================
# TOKEN EXTRACTION  (SDXL dual-tokenizer aware)
# ===============================
# def get_token_indices(tokenizer, prompt: str, words_to_track: List[str]):
#     """
#     Find token indices for specific words in the prompt.
#     Uses the first tokenizer (CLIP-L) — indices are consistent with
#     the cross-attention maps produced by the UNet's text_encoder path.
#     """
#     tokens     = tokenizer.encode(prompt)
#     token_strs = [tokenizer.decode([t]).strip() for t in tokens]

#     print(f"\nAll tokens: {list(enumerate(token_strs))}")

#     indices = []
#     for word in words_to_track:
#         word_lower = word.lower().strip()
#         found      = False
#         for i, tok in enumerate(token_strs):
#             if word_lower in tok.lower():
#                 indices.append(i)
#                 print(f"Found '{word}' at index {i}: '{tok}'")
#                 found = True
#                 break
#         if not found:
#             print(f"WARNING: '{word}' not found in tokens")

#     return indices


In [ ]:
def get_token_indices(tokenizer, prompt, words_to_track):
    token_ids = tokenizer.encode(prompt)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print("All tokens:", list(enumerate(tokens)))

    indices = []

    for word in words_to_track:
        word_tokens = tokenizer.tokenize(word)

        found = False
        for i in range(len(tokens) - len(word_tokens) + 1):
            if tokens[i:i + len(word_tokens)] == word_tokens:
                indices.append(i)
                print(f"Found '{word}' at index {i}")
                found = True
                break

        if not found:
            print(f"WARNING: '{word}' not found")

    return indices

In [ ]:
imagenet_templates = [
    "a photo of a {}.",
    "a photo of the {}.",
    "a rendering of a {}.",
    "a cropped photo of the {}.",
    "a close-up photo of a {}.",
    "a close-up photo of the {}.",
    "a bright photo of a {}.",
    "a dark photo of a {}.",
    "a photo of a clean {}.",
    "a photo of a dirty {}.",
    "a high resolution photo of a {}.",
    "a blurry photo of a {}.",
    "a detailed photo of a {}.",
    "a low resolution photo of a {}.",
    "a photo of a small {}.",
    "a photo of a large {}.",
    "a photo of the clean {}.",
    "a photo of the dirty {}.",
    "a photo of the shiny {}.",
    "a photo of the old {}.",
    "a photo of the new {}.",
    "a good photo of a {}.",
    "a bad photo of a {}.",
    "a studio photo of a {}.",
    "a photo of a {} in a room.",
    "a photo of a {} on the street.",
    "a photo of a {} on a table.",
    "a photo of a {} in the wild.",
    "a natural photo of a {}.",
    "a professional photo of a {}.",
    "a photo of a {} with natural lighting.",
    "a photo of a {} with dramatic lighting.",
    "a photo of a {} in daylight.",
    "a photo of a {} at night.",
    "a photo of a {} in the morning.",
    "a photo of a {} in the evening.",
    "a photo of a {} in the sunlight.",
    "a photo of a {} in the shadow.",
    "a photo of a {} outdoors.",
    "a photo of a {} indoors.",
    "a realistic photo of a {}.",
    "a stylized photo of a {}.",
    "an artistic photo of a {}.",
    "a minimalistic photo of a {}.",
    "a photo of a {} on a plain background.",
    "a photo of a {} on a white background.",
    "a photo of a {} on a black background.",
    "a photo of a {} on a colorful background.",
    "a photo of a {} in a natural environment.",
    "a photo of a {} in a forest.",
    "a photo of a {} in a city.",
    "a photo of a {} in a park.",
    "a photo of a {} near water.",
    "a photo of a {} on grass.",
    "a photo of a {} in motion.",
    "a photo of a {} standing still.",
    "a photo of a {} from the side.",
    "a photo of a {} from the front.",
    "a photo of a {} from above.",
    "a photo of a {} from below.",
    "a photo of a {} at a distance.",
    "a close view of a {}.",
    "a macro photo of a {}.",
    "a zoomed-in photo of a {}.",
    "a wide-angle photo of a {}.",
    "a photo of multiple {}.",
    "a photo of a single {}.",
    "a photo of the {} in nature.",
    "a photo of the {} on a surface.",
    "a photo of the {} in a landscape.",
    "a photo of the {} in a scene.",
    "a photograph of a {}.",
    "a photograph of the {}.",
    "a clear photo of a {}.",
    "a sharp photo of a {}.",
    "a slightly blurry photo of a {}.",
    "a photo of a {} with soft lighting.",
    "a photo of a {} with harsh lighting.",
    "a high-quality photo of a {}.",
    "a low-quality photo of a {}.",
]

In [ ]:
def get_embedding_for_prompt(model, prompt, templates):
    with torch.no_grad():
        texts      = [t.format(prompt) for t in templates]
        tokens     = clip.tokenize(texts).to(clip_device)
        embeddings = model.encode_text(tokens)
        embeddings /= embeddings.norm(dim=-1, keepdim=True)
        embedding  = embeddings.mean(dim=0)
        embedding  /= embedding.norm()
    return embedding.float()

In [ ]:
@torch.no_grad()
def clip_prompt_similarities(image: Image.Image, prompt: str):
    """Compute CLIP similarity between image and prompt parts."""
    if " and " in prompt:
        parts = prompt.split(" and ")
        p1    = parts[0].strip()
        p2    = " and ".join(parts[1:]).strip()
    else:
        p1 = p2 = prompt

    image_tensor = clip_preprocess(image).unsqueeze(0).to(clip_device)
    image_feat   = clip_model.encode_image(image_tensor)
    image_feat  /= image_feat.norm(dim=-1, keepdim=True)
    image_feat   = image_feat.float()

    full_feat = get_embedding_for_prompt(clip_model, prompt, imagenet_templates)
    p1_feat   = get_embedding_for_prompt(clip_model, p1,     imagenet_templates)
    p2_feat   = get_embedding_for_prompt(clip_model, p2,     imagenet_templates)

    return {
        "full_text":   (image_feat @ full_feat.T).item(),
        "first_half":  (image_feat @ p1_feat.T).item(),
        "second_half": (image_feat @ p2_feat.T).item(),
        "min_half":    min((image_feat @ p1_feat.T).item(),
                           (image_feat @ p2_feat.T).item()),
    }


In [ ]:
def generate(
    prompt: str,
    words_to_track: List[str],
    seed: int                  = 84,
    num_steps: int             = 50,
    guidance: float            = 9.5,
    height: int                = 768,
    width: int                 = 768,
    max_iter_to_alter: int     = 30,
    attend_excite_lr: float    = 0.2,
    alpha: float               = 10.0,
    spatial_loss_weight: float = 0.5,
    negative_prompt: str       = "",
):
    print(f"\n{'='*60}")
    print(f"Prompt    : {prompt}")
    print(f"Tracking  : {words_to_track}")
    print(f"Fuzzy loss: active for first {max_iter_to_alter} steps")
    print(f"{'='*60}")

    cfg = RunConfig(
        prompt              = prompt,
        seeds               = [seed],
        n_inference_steps   = num_steps,
        guidance_scale      = guidance,
        height              = height,
        width               = width,
        max_iter_to_alter   = max_iter_to_alter,
        attend_excite_lr    = attend_excite_lr,
        alpha               = alpha,
        spatial_loss_weight = spatial_loss_weight,
    )

    # SDXL has two tokenizers; tokenizer == CLIP-L (tokenizer_2 == OpenCLIP-G)
    # Token indices from tokenizer match the cross-attn maps captured by our processor
    token_indices = get_token_indices(model.tokenizer, prompt, words_to_track)

    if len(token_indices) == 0:
        print("ERROR: No tokens found!")
        return None, None, None

    print(f"Token indices: {token_indices}")

    attention_store = AttentionStore(attn_res=16)
    model.register_attention_control(attention_store)

    generator = torch.Generator(device='cuda' if torch.cuda.is_available() else 'cpu').manual_seed(seed)

    print("\nGenerating image with active fuzzy loss...\n")

    images = model(
        prompt              = prompt,
        token_indices       = token_indices,
        attention_store     = attention_store,
        num_inference_steps = num_steps,
        guidance_scale      = guidance,
        height              = height,
        width               = width,
        generator           = generator,
        output_type         = "pil",
        cfg                 = cfg,
        negative_prompt     = negative_prompt,
    )

    image = images[0]

    attention_scores = model.get_token_attention(attention_store, token_indices)
    clip_scores      = clip_prompt_similarities(image, prompt)

    print(f"\n{'='*60}")
    print("ATTENTION SCORES (post-generation):")
    for word, idx in zip(words_to_track, token_indices):
        score = attention_scores.get(idx, 0.0)
        print(f"  {word:15s} (token {idx:2d}): {score:.4f}")

    print(f"\n{'='*60}")
    print("CLIP SCORES:")
    print(f"  Full prompt   : {clip_scores['full_text']:.4f}")
    print(f"  First part    : {clip_scores['first_half']:.4f}")
    print(f"  Second part   : {clip_scores['second_half']:.4f}")
    print(f"  Min component : {clip_scores['min_half']:.4f}")
    print(f"{'='*60}\n")

    return image, clip_scores, attention_store


In [ ]:
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {device}")

In [ ]:
# Load VAE to CPU first; enable_model_cpu_offload will handle device placement
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch_dtype,
)

In [ ]:
model = FuzzyAttendExciteSDXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae            = vae,
    torch_dtype    = torch_dtype,
    variant        = "fp16" if torch.cuda.is_available() else None,
    use_safetensors= True,
)


In [ ]:
model.enable_model_cpu_offload()
model.enable_attention_slicing(slice_size=1)
model.enable_vae_tiling()

# ── 4. Swap scheduler to DPM++ ───────────────────────────────────────
model.scheduler = DPMSolverMultistepScheduler.from_config(
    model.scheduler.config,
    algorithm_type    = "dpmsolver++",
    use_karras_sigmas = True,
)

print("✓ SDXL Base model loaded with memory optimisations")

# Stage 2
    Build soft object masks from cross-attention.
    Stable for SDXL Base + Refiner predicate control.

In [ ]:
def build_object_masks(attn_store, token_indices, device, latent_size=128):

    avg = attn_store.get_average_attention()
    masks = {}

    for token_idx in token_indices:
        collected = []

        for key in ("down_cross", "mid_cross", "up_cross"):
            if key not in avg:
                continue

            attn = avg[key]  # [B, spatial, tokens]
            if token_idx >= attn.shape[-1]:
                continue

            # Select token-specific attention
            token_map = attn[0, :, token_idx]  # [spatial]

            spatial = int(token_map.shape[0] ** 0.5)
            if spatial * spatial != token_map.shape[0]:
                continue

            token_map = token_map.view(1, 1, spatial, spatial)
            collected.append(token_map)

        if not collected:
            continue

        # Aggregate FIRST (before normalization)
        token_map = torch.cat(collected, dim=0).mean(0, keepdim=True)

        # Remove very low background noise
        token_map = token_map - token_map.min()

        max_val = token_map.max()
        if max_val > 1e-8:
            token_map = token_map / max_val

        # 🔥 Resize to latent size
        token_map = F.interpolate(
            token_map,
            size=(latent_size, latent_size),
            mode="bilinear",
            align_corners=False,
        )[0, 0]

        # Light sharpening (optional but useful)
        token_map = token_map.pow(1.5)

        masks[token_idx] = token_map.clamp(0, 1).to(device)

        print(
            f"Token {token_idx}: "
            f"min={token_map.min():.4f} "
            f"max={token_map.max():.4f} "
            f"mean={token_map.mean():.4f}"
        )

    return masks

In [ ]:
class PredicateRefinerProcessor:

    def __init__(self, masks, strength=2.5):
        self.masks    = masks
        self.strength = strength

    def __call__(
        self, attn, hidden_states,
        encoder_hidden_states=None, attention_mask=None,
        temb=None, **kwargs          # ← absorb SDXL extra kwargs
    ):
        residual   = hidden_states
        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            B, C, H, W = hidden_states.shape
            hidden_states = hidden_states.view(B, C, H * W).transpose(1, 2)

        is_cross = encoder_hidden_states is not None
        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states

        query = attn.to_q(hidden_states)
        key   = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        query = attn.head_to_batch_dim(query)
        key   = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)

        attention_probs = attn.get_attention_scores(query, key, attention_mask)

        if is_cross:
            B_heads, N, T = attention_probs.shape
            spatial       = int(N ** 0.5)

            if spatial * spatial == N:
                for token_idx, full_mask in self.masks.items():
                    if token_idx >= T:
                        continue

                    mask = F.interpolate(
                        full_mask.unsqueeze(0).unsqueeze(0),
                        size=(spatial, spatial),
                        mode="bilinear",
                        align_corners=False,
                    )[0, 0].reshape(1, N, 1).repeat(B_heads, 1, 1)

                    attention_probs[:, :, token_idx:token_idx+1] *= (1 + self.strength * mask)
                    attention_probs[:, :, token_idx:token_idx+1] *= (1 - 0.7 * (1 - mask))

                attention_probs = attention_probs / (
                    attention_probs.sum(dim=-1, keepdim=True) + 1e-8)

        hidden_states = torch.bmm(attention_probs, value)
        hidden_states = attn.batch_to_head_dim(hidden_states)
        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(B, C, H, W)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor
        return hidden_states

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline


def load_refiner(device):
    refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-refiner-1.0",
        torch_dtype    = torch_dtype,
        variant        = "fp16" if device.type == "cuda" else None,
        use_safetensors= True,
    ).to(device)
    return refiner

In [ ]:
def attach_predicate_control(refiner, masks, strength=2.5):
    attn_procs = {}
    for name in refiner.unet.attn_processors.keys():
        if "attn2" in name:   # cross-attention in SDXL
            attn_procs[name] = PredicateRefinerProcessor(masks, strength=strength)
        else:
            attn_procs[name] = refiner.unet.attn_processors[name]
    refiner.unet.set_attn_processor(attn_procs)

In [ ]:
def refine_with_predicate(image, prompt, attn_store, token_indices):
    print("\nStarting Predicate-Aware Refinement...")

    refiner = load_refiner(device)

    # SDXL latents are 128×128
    masks = build_object_masks(attn_store, token_indices, device, latent_size=128)

    if len(masks) == 0:
        print("WARNING: No masks built — check token indices and attention store")
        return image

    print("Object masks built for tokens:", list(masks.keys()))
    for idx, m in masks.items():
        print(f"  Token {idx}: mask coverage = {m.mean().item():.3f}")

    attach_predicate_control(refiner, masks)

    refined = refiner(
        prompt             = prompt,
        image              = image,
        strength           = 0.4,
        num_inference_steps= 30,
        guidance_scale     = 9.5,
    ).images[0]

    clip_scores = clip_prompt_similarities(refined, prompt)

    print(f"\n{'='*60}")
    print("CLIP SCORES (refined):")
    print(f"  Full prompt    : {clip_scores['full_text']:.4f}")
    print(f"  First part     : {clip_scores['first_half']:.4f}")
    print(f"  Second part    : {clip_scores['second_half']:.4f}")
    print(f"  Min component  : {clip_scores['min_half']:.4f}")
    print(f"{'='*60}\n")

    return refined

In [ ]:
import torch
from pathlib import Path

# ===============================
# ENTRY POINT
# ===============================
if __name__ == "__main__":
    from IPython.display import display

    # Updated Configuration
    prompts = (
        "A very fast car"
    )
    
    words = ["fast", "car"]
    
    # Updated Negative Prompt for photorealism
    negative_prompt = (
        "(cartoon, illustration, animation, 2D, anime), painting, drawing, sketch, (CGI, 3D render, unreal engine, octane render), glossy plastic, smooth textures, vibrant saturated colors, neon, clean edges, flat lighting, vector artl, cel shading, flat shading, blurry, low quality, toy-like, neon colors, oversaturated colors, digital art, clipart, border, frame, (text, watermark)."                # Standard quality cleanup
    )

    # Define the list of seeds you want to test
    seeds = [142] 
    
    output_path = Path("./Fuzzy")
    output_path.mkdir(exist_ok=True)

    for current_seed in seeds:
        print(f"--- Generating for Seed: {current_seed} ---")
        
        image, scores, attn_store = generate(
            prompt              = prompts,
            words_to_track      = words,
            seed                = current_seed,
            num_steps           = 50,
            guidance            = 9.5,
            height              = 768, 
            width               = 768, 
            max_iter_to_alter   = 30,
            attend_excite_lr    = 0.2,
            alpha               = 10.0,
            spatial_loss_weight = 1.2,
            negative_prompt     = negative_prompt,
        )

        

        if image is not None:
            token_indices = get_token_indices(model.tokenizer, prompts, words)
            refined = refine_with_predicate(image, prompts, attn_store, token_indices)

            # Save files using the seed value in the filename
            image.save(output_path / f"seed_{current_seed}_stage1.png")
            refined.save(output_path / f"seed_{current_seed}_stage2_refined.png")

            print(f"✓ Images for seed {current_seed} saved to {output_path}")

            try:
                display(refined)
            except Exception:
                pass

    print("\nBatch Generation Complete.")